# 05. Variance partitioning

This notebook evaluates context-specific genomic relationship matrices for ACRs, UMRs, proximal TEs, their union, and the genome-wide reference. It combines the GSL spatial panel with LDAK REML results to produce Figure 4.


# 05. 方差分解

本 notebook 评估 ACRs、UMRs、proximal TEs、它们的 union 以及 genome-wide reference 的 context-specific genomic relationship matrices，并将 GSL spatial panel 与 LDAK REML 结果合并生成 Figure 4。


## Remote kernel initialization

The notebook runs against the remote project configuration. Verify the reported host, working directory, Python executable, environment, and expected kernelspec before continuing. This cell does not download data or start an analysis.


## 远程 kernel 初始化

本 notebook 根据远程项目配置运行。继续之前，请核对输出的 host、工作目录、Python executable、environment 和预期 kernelspec。本单元不会下载数据或启动分析。


In [ ]:
import os
import socket
import sys
from pathlib import Path

import yaml

%load_ext autoreload
%autoreload 2

launch_dir = Path.cwd().resolve()
project_file = launch_dir / "config" / "project.yaml"
if not project_file.is_file() and launch_dir.name == "notebooks":
    project_file = launch_dir.parent / "config" / "project.yaml"
if not project_file.is_file():
    raise FileNotFoundError("Copy config/project.example.yaml to config/project.yaml and complete it before running this notebook")
with project_file.open("r", encoding="utf-8") as stream:
    project_settings = yaml.safe_load(stream)
PROJECT_ROOT = Path(project_settings["remote"]["project_root"]).expanduser().resolve()
allowed_launch_dirs = {PROJECT_ROOT, PROJECT_ROOT / "notebooks"}
assert launch_dir in allowed_launch_dirs, f"Start Jupyter from the configured remote project root: {PROJECT_ROOT}"
assert PROJECT_ROOT.is_dir(), f"Remote project does not exist: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
expected_environment = project_settings["jupyter"]["kernel"]["environment"]["name"]
active_environment = os.environ.get("CONDA_DEFAULT_ENV", Path(sys.prefix).name)
if project_settings["jupyter"]["same_environment"]:
    server_environment = project_settings["jupyter"]["server"]["environment"]["name"]
    assert server_environment == expected_environment, (server_environment, expected_environment)
assert active_environment == expected_environment, (active_environment, expected_environment)
print({
    "host": socket.gethostname(),
    "working_directory": str(Path.cwd()),
    "python_executable": sys.executable,
    "environment": active_environment,
    "expected_kernelspec": project_settings["jupyter"]["kernel"]["kernelspec"],
})


## Run mode selection

Use `checkpoint` to read the published intermediate inputs or `full` to rebuild this notebook's intermediate products from source data. Outputs from the current run are always written to generated directories and never overwrite reference files.


## 运行模式选择

使用 `checkpoint` 读取已发布的中间输入；使用 `full` 从 source data 重建本 notebook 的中间产品。本次运行的输出始终写入 generated 目录，不覆盖 reference 文件。


In [ ]:
RUN_MODE = "checkpoint"
if RUN_MODE not in {"checkpoint", "full"}:
    raise ValueError("RUN_MODE must be checkpoint or full")
RUN_MODE


## Inputs and output contracts

All scientific inputs are resolved below `PROJECT_ROOT/data`. Published inputs are read from source or reference directories; newly built checkpoints and results are written only to generated directories. The following sections validate the notebook-specific schemas before interpreting results.


## 输入与输出契约

所有科学输入均从 `PROJECT_ROOT/data` 下解析。已发布输入从 source 或 reference 目录读取；新建 checkpoints 和 results 只写入 generated 目录。以下章节在解释结果前检查本 notebook 特有的 schemas。


In [ ]:
import sys
import warnings
from functools import reduce
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from laenrich.config import load_project_paths
from laenrich.io import read_table, validate_analysis_mode
from laenrich.plotting import save_figure
from laenrich.variance import parse_ldak_reml, summarize_matched_null, validate_grm

paths = load_project_paths(PROJECT_ROOT / "config" / "project.yaml")
project_dir = paths.project_root
mode = validate_analysis_mode(RUN_MODE)
checkpoint_dir = paths.checkpoint_reference_dir / "variance" if mode == "checkpoint" else paths.checkpoint_generated_dir / "variance"
figure_data_dir = paths.result_generated_dir / "figure_data" / "figure4"
figure_dir = paths.result_generated_dir / "figures"
supplementary_table_dir = paths.result_generated_dir / "supplementary_tables"
audit_dir = paths.result_generated_dir / "audits" / "05_variance"
spatial_dir = figure_data_dir
for directory in [figure_data_dir, figure_dir, supplementary_table_dir, audit_dir]:
    directory.mkdir(parents=True, exist_ok=True)
print({"mode": mode, "checkpoint_dir": str(checkpoint_dir)})


## 1. Variant sets and matched controls

The focal sets are ACRs, UMRs, proximal TEs, their union, and all eligible genome-wide variants. Each null set has the same site count and is matched by MAF, recombination rate, nucleotide diversity, and conditional scope.


## 1. Variant sets 与匹配对照

Focal sets 包括 ACRs、UMRs、proximal TEs、它们的 union 以及全部 eligible genome-wide variants。每个 null set 具有相同位点数，并匹配 MAF、recombination rate、nucleotide diversity 和 conditional scope。


In [ ]:
variant_sets = read_table(
    checkpoint_dir / "variant_set_manifest.tsv",
    ["set_id", "context", "set_type", "replicate", "n_variants", "variant_file"],
)
focal_counts = variant_sets.loc[variant_sets["set_type"].eq("context")].set_index("context")["n_variants"]
control_counts = variant_sets.loc[variant_sets["set_type"].eq("matched_control")]
assert control_counts.apply(lambda row: row["n_variants"] == focal_counts.loc[row["context"]], axis=1).all()
matching_audit = read_table(
    checkpoint_dir / "matched_control_audit.tsv",
    ["set_id", "variable", "standardized_mean_difference", "same_scope", "overlap_with_context"],
)
assert matching_audit["same_scope"].all()
assert matching_audit["overlap_with_context"].eq(0).all()
variant_sets.groupby(["context", "set_type"])["set_id"].nunique()


## 2. GRM construction and validation

All GRMs use the same accession order. Symmetry, dimensions, diagonal scale, finite values, and minimum eigenvalues are checked before REML. The genome-wide GRM is a reference, not the permutation null.


## 2. GRM 构建与验证

所有 GRMs 使用相同样本顺序。REML 前检查 symmetry、dimensions、diagonal scale、finite values 和 minimum eigenvalues。Genome-wide GRM 是参照，而不是 permutation null。


In [ ]:
grm_records = []
grm_manifest = read_table(checkpoint_dir / "grm_manifest.tsv", ["set_id", "matrix_npy", "sample_ids"])
for row in grm_manifest.itertuples(index=False):
    matrix_path = Path(row.matrix_npy)
    sample_path = Path(row.sample_ids)
    if not matrix_path.is_absolute():
        matrix_path = checkpoint_dir / matrix_path
    if not sample_path.is_absolute():
        sample_path = checkpoint_dir / sample_path
    matrix = np.load(matrix_path)
    sample_ids = pd.Index(pd.read_csv(sample_path, sep="\t", header=None).iloc[:, 0].astype(str))
    grm_records.append({"set_id": row.set_id, **validate_grm(matrix, sample_ids)})
grm_qc = pd.DataFrame.from_records(grm_records)
grm_qc.head()


## 3. LDAK REML, union, and multi-GRM models

Separate REML models estimate variance explained by context, matched-control, and genome-wide GRMs for GSL, PHQ, AL, and STR. Union and multi-GRM models test combined regulatory contexts without counting their overlap twice.


## 3. LDAK REML、union 与 multi-GRM models

Separate REML models 估计 context、matched-control 和 genome-wide GRMs 对 GSL、PHQ、AL 和 STR 的方差解释。Union 和 multi-GRM models 检验组合 regulatory contexts，同时避免重复计算重叠位点。


In [ ]:
reml_manifest = read_table(
    checkpoint_dir / "reml_manifest.tsv",
    ["set_id", "context", "set_type", "replicate", "variable_id", "reml_file"],
)
reml_records = []
for row in reml_manifest.itertuples(index=False):
    reml_path = Path(row.reml_file)
    if not reml_path.is_absolute():
        reml_path = checkpoint_dir / reml_path
    estimates = parse_ldak_reml(reml_path)
    heritability = estimates.loc[estimates["parameter"].str.lower().isin(["heritability", "her_all", "her_k1"])].iloc[0]
    reml_records.append({"set_id": row.set_id, "context": row.context, "set_type": row.set_type, "replicate": row.replicate, "variable_id": row.variable_id, "variance_explained": heritability["estimate"], "standard_error": heritability["standard_error"]})
reml_results = pd.DataFrame.from_records(reml_records)
required_contexts = {"acr", "umr", "proximal_te", "regulatory_union", "genome"}
assert required_contexts.issubset(set(reml_results["context"]))
reml_results.groupby(["context", "set_type"])["variance_explained"].describe()


## 4. Empirical inference and Figure 4

For each context and environmental variable, the observed variance explained is compared with 1,000 matched-control GRMs. Figure 4 combines the spatial GSL profile from notebook 04 with context-specific variance estimates.


## 4. Empirical inference 与 Figure 4

对每个 context 和环境变量，将 observed variance explained 与 1,000 个 matched-control GRMs 比较。Figure 4 合并 notebook 04 的 spatial GSL profile 和 context-specific variance estimates。


In [ ]:
summary_records = []
for (context, variable_id), group in reml_results.groupby(["context", "variable_id"], sort=True):
    observed_rows = group.loc[group["set_type"].eq("context"), "variance_explained"]
    controls = group.loc[group["set_type"].eq("matched_control"), "variance_explained"]
    if len(observed_rows) == 1 and len(controls) > 0:
        summary_records.append({"context": context, "variable_id": variable_id, **summarize_matched_null(float(observed_rows.iloc[0]), controls)})
variance_summary = pd.DataFrame.from_records(summary_records)
spatial_profiles = pd.read_hdf(spatial_dir / "spatial_profiles.h5", key="df")
gsl_profile = spatial_profiles.loc[
    spatial_profiles["variable_id"].eq("gsl") & spatial_profiles["anchor"].eq("tss") & spatial_profiles["track"].eq("envgwas_signal")
]
figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(gsl_profile["window_center"], gsl_profile["relative_signal"], color="#173f5f")
axes[0].fill_between(gsl_profile["window_center"], gsl_profile["null_lower"], gsl_profile["null_upper"], color="0.8")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Position relative to TSS (bp)")
axes[0].set_ylabel("Relative envGWAS signal")
plot_data = variance_summary.loc[variance_summary["variable_id"].eq("gsl")].sort_values("context")
axes[1].bar(plot_data["context"], plot_data["observed"], color="#2f4b7c")
axes[1].scatter(plot_data["context"], plot_data["null_median"], color="#b22222", zorder=3)
axes[1].set_ylabel("Variance explained")
axes[1].tick_params(axis="x", rotation=35)
figure.tight_layout()
save_figure(figure, figure_dir / "figure4.pdf")
plt.close(figure)
reml_results.to_csv(supplementary_table_dir / "table_s4_variance_partitioning.tsv", sep="\t", index=False)
variance_summary.to_csv(figure_data_dir / "variance_partitioning_summary.tsv", sep="\t", index=False)
grm_qc.to_csv(audit_dir / "grm_qc.tsv", sep="\t", index=False)
variant_sets.to_csv(audit_dir / "grm_metadata.tsv", sep="\t", index=False)


## Manual recomputation command

Variant-set GRMs and REML estimates are built outside the notebook for each context, matched replicate, union, genome-wide reference, and multi-GRM model:

[SCHEDULER JOB]

```bash
bash src/run_ldak_reml.sh --ldak <ldak-executable> --bfile <plink-prefix> --phenotype <phenotype-file> --output-prefix <reml-output-prefix> [--mgrm <multi-grm-list>]
```

The notebook reads only completed GRM metadata and audited REML outputs.


## 手动重算命令

每个 context、matched replicate、union、genome-wide reference 和 multi-GRM model 的 variant-set GRMs 与 REML estimates 均在 notebook 外构建：

[SCHEDULER JOB]

```bash
bash src/run_ldak_reml.sh --ldak <ldak-executable> --bfile <plink-prefix> --phenotype <phenotype-file> --output-prefix <reml-output-prefix> [--mgrm <multi-grm-list>]
```

本 notebook 只读取已完成的 GRM metadata 和经过审计的 REML outputs。


## Saved outputs and next step

Generated files remain in the remote working directory. Inspect the reported row counts, identifiers, quality-control summaries, and figure panels before proceeding to the next numbered notebook. External-tool commands described above remain manual steps.


## 已保存输出与下一步

生成的文件保留在远程工作目录中。进入下一本编号 notebook 前，请检查报告的行数、标识符、quality-control summaries 和 figure panels。以上 external-tool commands 仍需人工执行。
